# Lab 6 - Coder Agent

A coder agent doesn't necessarily mean that it generates code - but more broadly that it can write code and run it in order to solve a task.

In [ ]:
from agents import Agent, Runner, trace, function_tool, ModelSettings, AsyncOpenAI, OpenAIChatCompletionsModel
import docker
import tempfile
import os
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [154]:
# Define your settings separately
# Use max_completion_tokens as it's the 2026 standard for reasoning models
settings = ModelSettings(
    temperature=0.0, # Reduce creativity to keep JSON stable
    max_completion_tokens=1024
)

In [155]:
client = docker.from_env()
image = "python:3.12-slim"

In [156]:
client.containers.run(image="python:3.12-slim", command=["python", "-c", "print(2+2)"], remove=True)

b'4\n'

In [163]:
@function_tool
def execute_python(code: str) -> str:
    """
    Execute the given Python code inside a Docker container with python:3.12-slim,
    and return whatever is printed to stdout.
    You must print the result of the code to stdout in order to retrieve it.
    This uses the python:3.12-slim image and so it does not have scientific libraries installed;
    write simple python 3.12 code using the standard library only. Do not use numpy or scipy.
    IMPORTANT: You must print the result of the code in order to retrieve it.

    Args:
        code: The Python code to run. Remember to print the result.

    """
    print(f"Executing code: {code}")
    with tempfile.TemporaryDirectory() as tmpdir:
        script_path = os.path.join(tmpdir, "script.py")
        volumes = {tmpdir: {"bind": "/tmp", "mode": "ro"}}
        command = ["python", "/tmp/script.py"]
        with open(script_path, "w") as f:
            f.write(code)
        logs = client.containers.run(image=image, command=command, volumes=volumes, remove=True)
    result = logs.decode("utf-8")
    print(f"Result: {result}")
    return result

In [158]:
print(execute_python.description)

Execute Python code. 
NOTE: Do not call this tool with the same code twice. 
If you already have the result, provide the final answer.


In [170]:
# The original instructions will not work. These do work.
instructions = """
You are a Coder Agent. 
1. Use 'execute_python' to find prime factors.
2. Once you have the result, you MUST respond ONLY with a JSON object.
3. DO NOT include any conversational text, markdown formatting (like ```json), or explanations outside the JSON.
4. The JSON must follow this exact format: {"factors": [numbers], "explanation": "text"}
"""

In [160]:
number = 5 * 11 * 47 * 307
input = f"What are the prime factors of {number}? Reply only with the answer."

In [161]:
from requests import api


external_client =  AsyncOpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
LLM = OpenAIChatCompletionsModel(model='gpt-oss:20b', openai_client=external_client )

In [169]:
agent = Agent("Coder Agent", model=LLM, instructions=instructions, model_settings=settings, tools=[execute_python])

with trace("Coder Agent"):
    result = await Runner.run(agent, input)
    print(result.final_output)

Executing code: n=793595
factors=[]
while n%2==0:
    factors.append(2)
    n//=2
p=3
while p*p<=n:
    while n%p==0:
        factors.append(p)
        n//=p
    p+=2
if n>1:
    factors.append(n)
print(factors)
Result: [5, 11, 47, 307]

{"factors":[5,11,47,307],"explanation":"793595 = 5 × 11 × 47 × 307"}


In [177]:
# The original instructions will not work. These do work.
instructions = """
You are a Financial Engineering Agent.
1. When given a pricing task, use 'execute_python' to define and run a Black-Scholes function.
2. After you receive the result from the tool, respond ONLY with a JSON object. 
3. DO NOT provide markdown code blocks, explanations, or Python assignments.

Follow this exact format for your final response:
{"option_price": <float>, "ticker": "GOOGL", "type": "call"}
"""

In [174]:


input = """
Write a python function to calculate the price of an option using the Black-Scholes model.
Then use your tool to execute the code and calculate the price of this option:
GOOGL
Stock Price: 150
Strike Price: 155
Time to maturity: 3 years
Risk free rate: 1%
Volatility: 20%
Dividend Yield: 0%
Respond with the calculated price of the option only.
"""

In [178]:

agent = Agent("Coder Agent", model=LLM, model_settings=settings, instructions=instructions, tools=[execute_python])

with trace("Coder Agent"):
    result = await Runner.run(agent, input)
    print(result.final_output)

Executing code: import math
S=150
K=155
T=3
r=0.01
sigma=0.20
q=0.0

def N(x):
    return 0.5*(1+math.erf(x/math.sqrt(2)))

d1=(math.log(S/K)+(r-q+0.5*sigma**2)*T)/(sigma*math.sqrt(T))
d2=d1-sigma*math.sqrt(T)
call=S*math.exp(-q*T)*N(d1)-K*math.exp(-r*T)*N(d2)
print(call)

Result: 20.44641206078066

{"option_price": 20.44641206078066, "ticker": "GOOGL", "type": "call"}


In [176]:
import math

def black_scholes_call(S, K, T, r, sigma, q=0.0):
    d1 = (math.log(S / K) + (r - q + 0.5 * sigma ** 2) * T) / (sigma * math.sqrt(T))
    d2 = d1 - sigma * math.sqrt(T)
    def N(x):
        return 0.5 * (1 + math.erf(x / math.sqrt(2)))
    return S * math.exp(-q * T) * N(d1) - K * math.exp(-r * T) * N(d2)

S = 150
K = 155
T = 3
r = 0.01
sigma = 0.20
q = 0.0
price = black_scholes_call(S, K, T, r, sigma, q)
print(price)

Result: 20.44641206078066

#{"option_price": 20.44641206078066, "ticker": "GOOGL", "type": "call"}

20.44641206078066
